In [14]:
import sys

! {sys.executable} -m pip install line_profiler

     |████████████████████████████████| 53 kB 453 kB/s eta 0:00:01
You should consider upgrading via the '/Users/bmac3/Code/ATAP/atap-roots-controls/venv/bin/python -m pip install --upgrade pip' command.


In this notebook we want to test the speed of different methods for reading in binary data

In [44]:
# setup for the notebook (importing libs and defining constants to be used)

from collections import namedtuple
from struct import Struct

import cbitstruct as bitstruct
from line_profiler import LineProfiler
import numpy as np
import pandas as pd

bin_file = "pixie16_binary_data.bin"

with open(bin_file, 'rb') as fp:
    bin_data = fp.read()
    
single_event_bin_data = bin_data[:16]
correct_bit_fields = pd.read_hdf('correct_bit_fields.hdf')

Let's make a list of the operation that need to be done to the data and think of ways we can combine/reorder/substitute the operations to make it faster.

1. get the data into memory
  * open a connection
    * read in all the bits at once
    * read in the bits in a loop
  * load the file directly into memory
    
2. parse the bits into their datatypes

3. combine the variables to get derived values

Let's get a sense of how long each individual operation will take.

In [2]:
# how long does it take to open a file?

def open_10_000():
    for _ in range(10_000):
        f = open(bin_file, "rb")


%timeit open_10_000()

183 ms ± 3.41 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


so 20.7 us for an open

In [3]:
# how long does it take to load in an entire file?

def load_bin_file():
    for _ in range(10_000):
        f = open(bin_file, "rb")
        r = f.read()

%timeit load_bin_file()

454 ms ± 4.82 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


so 49.1 us for open and read. This should translate to 49.1 - 20.7 = 28.4 us for read

In [4]:
# check if the previous conclusion is true with line_reader.

lp = LineProfiler(load_bin_file)
lp.run("load_bin_file()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.49275 s
File: <ipython-input-3-014c27ea96fb>
Function: load_bin_file at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def load_bin_file():
     4     10001       4198.0      0.4      0.9      for _ in range(10_000):
     5     10000     221887.0     22.2     45.0          f = open(bin_file, "rb")
     6     10000     266665.0     26.7     54.1          r = f.read()



26.5 us for read and 24.8 us for open. This looks about right.

In [5]:
# see how long this takes loading bin file into the local namespace first

def load_bin_file_local_namespace():
    global bin_file
    for _ in range(10_000):
        f = open(bin_file, "rb")
        r = f.read()

%timeit load_bin_file()

464 ms ± 11.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Even longer (no appreciable difference)

In [6]:
# see if using a context manager will change the runtime significantly

def load_bin_file_with_context_manager():
    for _ in range(10_000):
        with open(bin_file, "rb") as f:
            r = f.read()

%timeit load_bin_file_with_context_manager()

417 ms ± 4.46 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


so 43.3 us for open and read. looks like it takes slightly less time to use a context manager.

In [7]:
# check if the previous conclusion is true with line_reader

lp = LineProfiler(load_bin_file_with_context_manager)
lp.run("load_bin_file_with_context_manager()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.449525 s
File: <ipython-input-6-44351f4afd5a>
Function: load_bin_file_with_context_manager at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def load_bin_file_with_context_manager():
     4     10001       4136.0      0.4      0.9      for _ in range(10_000):
     5     10000     151493.0     15.1     33.7          with open(bin_file, "rb") as f:
     6     10000     293896.0     29.4     65.4              r = f.read()



Interestingly the opening operation looks like it takes less time with the context manager

We need to read in the loaded data as individual events. In order to do this we need a way of partitioning our data into data for each event. Let's assume we don't have traces and that each event is only the header.

In [8]:
# verify that our assumption is true for our sample data

print(correct_bit_fields['Event Length'].unique())
print(correct_bit_fields['Trace Length'].unique())

[4]
[0]


We see that the event length for every event is 4 so indeed we have only headers

In [26]:
# how long bitstruct will take for a single event?

HEADER_FORMATS = {
    0: (
        "b1u14u5u4u4u4",
        ("Finish Code", "Event Length", "Header Length", "CrateID", "SlotID", "Chan#"),
    ),
    1: ("u32", ("EVTTIME_LO",)),
    2: ("u3u13u16", ("CFD trigger source bits", "CFD Fractional Time", "EVTTIME_HI")),
    3: ("b1u15u16", ("Trace Out-of-Range Flag", "Trace Length", "Event Energy")),
}

bit_fmt = ""
field_names = ()

for i in HEADER_FORMATS.keys():
    bit_fmt += HEADER_FORMATS[i][0]
    field_names += HEADER_FORMATS[i][1]

def parse_binary_event_with_bitstruct():
    for _ in range(10_000):
        header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
        event_fields = bitstruct.unpack(bit_fmt, header_bits)
        
        
lp = LineProfiler(parse_binary_event_with_bitstruct)
lp.run("parse_binary_event_with_bitstruct()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.021897 s
File: <ipython-input-26-e34f9f833786>
Function: parse_binary_event_with_bitstruct at line 20

Line #      Hits         Time  Per Hit   % Time  Line Contents
    20                                           def parse_binary_event_with_bitstruct():
    21     10001       3728.0      0.4     17.0      for _ in range(10_000):
    22     10000       7342.0      0.7     33.5          header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
    23     10000      10827.0      1.1     49.4          event_fields = bitstruct.unpack(bit_fmt, header_bits)



So it takes on average 31.8 + 51.1 = 82.9 us to parse an event this way with bitstruct.

In [58]:
# how long does it take with a bitstruct compiled object?

cf = bitstruct.compile(bit_fmt)


def parse_binary_event_with_bitstruct_compiled():
    for _ in range(10_000):
        header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
        event_fields = cf.unpack(header_bits)
        
        
lp = LineProfiler(parse_binary_event_with_bitstruct_compiled)
lp.run("parse_binary_event_with_bitstruct_compiled()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.017054 s
File: <ipython-input-58-8f80dbcb9bef>
Function: parse_binary_event_with_bitstruct_compiled at line 6

Line #      Hits         Time  Per Hit   % Time  Line Contents
     6                                           def parse_binary_event_with_bitstruct_compiled():
     7     10001       3158.0      0.3     18.5      for _ in range(10_000):
     8     10000       6410.0      0.6     37.6          header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
     9     10000       7486.0      0.7     43.9          event_fields = cf.unpack(header_bits)



This cuts the unpack time from 1.1us to 0.7us.

In [59]:
# see how long this takes if we want named tuples


field_names = list(map(lambda s: s.replace(' ', '_').replace('#', '').replace('-', '_'), field_names))
Event = namedtuple('Event', field_names)


def parse_binary_event_with_bitstruct_and_namedtuples():
    for _ in range(10_000):
        header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
        event_fields = cf.unpack(header_bits)
        event = Event(*event_fields)
        
lp = LineProfiler(parse_binary_event_with_bitstruct_and_namedtuples)
lp.run("parse_binary_event_with_bitstruct_and_namedtuples()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.028859 s
File: <ipython-input-59-0e133373b42a>
Function: parse_binary_event_with_bitstruct_and_namedtuples at line 8

Line #      Hits         Time  Per Hit   % Time  Line Contents
     8                                           def parse_binary_event_with_bitstruct_and_namedtuples():
     9     10001       3288.0      0.3     11.4      for _ in range(10_000):
    10     10000       6581.0      0.7     22.8          header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
    11     10000       7478.0      0.7     25.9          event_fields = cf.unpack(header_bits)
    12     10000      11512.0      1.2     39.9          event = Event(*event_fields)



Conversion to a namedtuple takes the longest of any operation here. Probably not worth including this in the bitstruct implementation

In [42]:
# how long will a bitstruct implementation with unpack_dict take?

def parse_binary_event_with_bitstruct_to_dict():
    for _ in range(10_000):
        header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
        event_fields = bitstruct.unpack_dict(bit_fmt, field_names, header_bits)
        
lp = LineProfiler(parse_binary_event_with_bitstruct_to_dict)
lp.run("parse_binary_event_with_bitstruct_to_dict()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.026878 s
File: <ipython-input-42-5355698fe944>
Function: parse_binary_event_with_bitstruct_to_dict at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def parse_binary_event_with_bitstruct_to_dict():
     4     10001       3291.0      0.3     12.2      for _ in range(10_000):
     5     10000       7127.0      0.7     26.5          header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
     6     10000      16460.0      1.6     61.2          event_fields = bitstruct.unpack_dict(bit_fmt, field_names, header_bits)



This method offers a (1.0 + 1.2) - 1.6 = 0.6us speed up over converting to a namedtuple, and a 1.6 - 1.1 = 0.5us slow down compared to the barebones bitstruct unpack.

In [60]:
# how much improvement do we get with compiled unpack dict?

cf_dict = bitstruct.compile(bit_fmt, field_names)

def parse_binary_event_with_bitstruct_to_dict_compiled():
    for _ in range(10_000):
        header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
        event_fields = cf_dict.unpack(header_bits)
        
lp = LineProfiler(parse_binary_event_with_bitstruct_to_dict_compiled)
lp.run("parse_binary_event_with_bitstruct_to_dict_compiled()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.023024 s
File: <ipython-input-60-3ee1f2865fe1>
Function: parse_binary_event_with_bitstruct_to_dict_compiled at line 5

Line #      Hits         Time  Per Hit   % Time  Line Contents
     5                                           def parse_binary_event_with_bitstruct_to_dict_compiled():
     6     10001       3430.0      0.3     14.9      for _ in range(10_000):
     7     10000       6423.0      0.6     27.9          header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
     8     10000      13171.0      1.3     57.2          event_fields = cf_dict.unpack(header_bits)



This method offers a 0.3us improvement over the uncomplied version, but still 0.4us slower than the compiled version of unpack

In [48]:
# how long will a struct / python bitwise arithmetic approach take?

HEADER = Struct("<2I4H")
SUMS = Struct("<3If")


def parse_binary_event_with_struct():
    for _ in range(10_000):
        pileup_bits, eventtime_lo, eventtime_hi, cfd_bits, event_energy, trace_bits = HEADER.unpack_from(single_event_bin_data)
        event_fields = (
            pileup_bits >> 31,
            (pileup_bits >> 17) & ((1 << 14) - 1),
            (pileup_bits >> 12) & 0b11111,
            (pileup_bits >> 8) & 0b1111,
            (pileup_bits >> 4) & 0b1111,
            pileup_bits & 0b1111,
            eventtime_lo,
            (cfd_bits >> 13) & 0b111,
            cfd_bits & ((1 << 13) - 1),
            eventtime_hi,
            trace_bits >> 15,
            trace_bits & ((1 << 16) - 1),
            event_energy
        )
        
lp = LineProfiler(parse_binary_event_with_struct)
lp.run("parse_binary_event_with_struct()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.074452 s
File: <ipython-input-48-565056de591f>
Function: parse_binary_event_with_struct at line 7

Line #      Hits         Time  Per Hit   % Time  Line Contents
     7                                           def parse_binary_event_with_struct():
     8     10001       4184.0      0.4      5.6      for _ in range(10_000):
     9     10000       7743.0      0.8     10.4          pileup_bits, eventtime_lo, eventtime_hi, cfd_bits, event_energy, trace_bits = HEADER.unpack_from(single_event_bin_data)
    10     10000       4553.0      0.5      6.1          event_fields = (
    11     10000       4134.0      0.4      5.6              pileup_bits >> 31,
    12     10000       5313.0      0.5      7.1              (pileup_bits >> 17) & ((1 << 14) - 1),
    13     10000       4622.0      0.5      6.2              (pileup_bits >> 12) & 0b11111,
    14     10000       5132.0      0.5      6.9              (pileup_bits >> 8) & 0b1111,
    15     10000       484

Each operation here takes ~0.5us, this is significantly slower than bitstruct

Let's take the best parts of what we have learned here and see if we can have success paralellizing the read in

In [61]:
# write our fastest version of reading a single event

def parse_single_binary_event(binary_data):
    header_bits = bitstruct.byteswap(4 * '4', binary_data)
    return cf.unpack(header_bits)

In [62]:
# how fast does it take to loop over all the data, appending each entry to a list?

def read_events_linear_with_list():
    all_events = []
    for i in range(len(bin_data) // 16):
        all_events.append(parse_single_binary_event(bin_data[i*16: (i+1)*16]))
    return all_events


lp = LineProfiler(read_events_linear_with_list)
lp.run("read_events_linear_with_list()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.055445 s
File: <ipython-input-62-7f621a5d9b08>
Function: read_events_linear_with_list at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def read_events_linear_with_list():
     4         1          1.0      1.0      0.0      all_events = []
     5     24599       8377.0      0.3     15.1      for i in range(len(bin_data) // 16):
     6     24598      47067.0      1.9     84.9          all_events.append(parse_single_binary_event(bin_data[i*16: (i+1)*16]))
     7         1          0.0      0.0      0.0      return all_events



In [64]:
# how fast does it take to loop over all the data and assigning to a preallocated list?

def read_events_linear_with_preallocated_list():
    n_events = len(bin_data) // 16
    all_events = n_events * [None]
    for i in range(n_events):
        all_events[i] = parse_single_binary_event(bin_data[i*16: (i+1)*16])
    return all_events


lp = LineProfiler(read_events_linear_with_preallocated_list)
lp.run("read_events_linear_with_preallocated_list()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.056589 s
File: <ipython-input-64-8c6a677f5638>
Function: read_events_linear_with_preallocated_list at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def read_events_linear_with_preallocated_list():
     4         1          1.0      1.0      0.0      n_events = len(bin_data) // 16
     5         1         12.0     12.0      0.0      all_events = n_events * [None]
     6     24599       8740.0      0.4     15.4      for i in range(n_events):
     7     24598      47835.0      1.9     84.5          all_events[i] = parse_single_binary_event(bin_data[i*16: (i+1)*16])
     8         1          1.0      1.0      0.0      return all_events



This is actually slower than not preallocating

In [102]:
# how long will it take to preallocate a pandas dataframe and then fill it

def read_events_linear_with_preallocated_df():
    n_events = len(bin_data) // 16
    all_events = pd.DataFrame({
        field_name: pd.Series(index=range(n_events), dtype=dtype) for field_name, dtype in zip(correct_bit_fields.columns, correct_bit_fields.dtypes)
    })
    for i in range(n_events):
        all_events.iloc[i] = parse_single_binary_event(bin_data[i*16: (i+1)*16])
    return all_events


lp = LineProfiler(read_events_linear_with_preallocated_df)
lp.run("read_events_linear_with_preallocated_df()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 106.15 s
File: <ipython-input-102-042baaa8d8a4>
Function: read_events_linear_with_preallocated_df at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def read_events_linear_with_preallocated_df():
     4         1          2.0      2.0      0.0      n_events = len(bin_data) // 16
     5         2       6739.0   3369.5      0.0      all_events = pd.DataFrame({
     6         1        352.0    352.0      0.0          field_name: pd.Series(index=range(n_events), dtype=dtype) for field_name, dtype in zip(correct_bit_fields.columns, correct_bit_fields.dtypes)
     7                                               })
     8     24599      39255.0      1.6      0.0      for i in range(n_events):
     9     24598  106103421.0   4313.5    100.0          all_events.iloc[i] = parse_single_binary_event(bin_data[i*16: (i+1)*16])
    10         1          1.0      1.0      0.0      return all_events

This takes ~4000 times longer than the list version.

In [103]:
# how long does it take to convert the list of tuples to a dataframe?

def read_events_linear_with_list_to_pandas():
    all_events = []
    for i in range(len(bin_data) // 16):
        all_events.append(parse_single_binary_event(bin_data[i*16: (i+1)*16]))
    return pd.DataFrame.from_records(all_events, columns=field_names)


lp = LineProfiler(read_events_linear_with_list_to_pandas)
lp.run("read_events_linear_with_list_to_pandas()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.112102 s
File: <ipython-input-103-f3f9b4e5c8cd>
Function: read_events_linear_with_list_to_pandas at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def read_events_linear_with_list_to_pandas():
     4         1          2.0      2.0      0.0      all_events = []
     5     24599       9661.0      0.4      8.6      for i in range(len(bin_data) // 16):
     6     24598      48541.0      2.0     43.3          all_events.append(parse_single_binary_event(bin_data[i*16: (i+1)*16]))
     7         1      53898.0  53898.0     48.1      return pd.DataFrame.from_records(all_events, columns=field_names)



Much faster this way, but the conversion to dataframe still takes a lot of time

Let's expore if there are options to read in the data from buffer with numpy

In [121]:
# how long does it take to read in with preallocated numpy array():

np_dtype = np.format_parser(formats='b1,i8,i8,i8,i8,i8,i8,i8,i8,i8,b1,i8,i8'.split(','), names=field_names, titles=[]).dtype


def read_events_linear_with_preallocated_numpy():
    n_events = len(bin_data) // 16
    all_events = np.empty(n_events, dtype=np_dtype)
    for i in range(n_events):
        all_events[i] = parse_single_binary_event(bin_data[i*16: (i+1)*16])
    return pd.DataFrame(all_events)


lp = LineProfiler(read_events_linear_with_preallocated_numpy)
lp.run("read_events_linear_with_preallocated_numpy()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.081004 s
File: <ipython-input-121-a214be144167>
Function: read_events_linear_with_preallocated_numpy at line 6

Line #      Hits         Time  Per Hit   % Time  Line Contents
     6                                           def read_events_linear_with_preallocated_numpy():
     7         1          1.0      1.0      0.0      n_events = len(bin_data) // 16
     8         1          8.0      8.0      0.0      all_events = np.empty(n_events, dtype=np_dtype)
     9     24599       9669.0      0.4     11.9      for i in range(n_events):
    10     24598      65032.0      2.6     80.3          all_events[i] = parse_single_binary_event(bin_data[i*16: (i+1)*16])
    11         1       6294.0   6294.0      7.8      return pd.DataFrame(all_events)



An 8.5x speedup converting to pandas dataframe but a 37% slowdown for the assignment

In [ ]:
# What kind of speed ups can we get using multiprocessing?



In [122]:
53898.0 / 6294.0

8.563393708293614

In [123]:
(2.6 - 1.9) / 1.9

0.36842105263157904

In [110]:
pd.DataFrame(read_events_linear_with_preallocated_numpy(), columns=field_names)

,Finish_Code,Event_Length,Header_Length,CrateID,SlotID,Chan,EVTTIME_LO,CFD_trigger_source_bits,CFD_Fractional_Time,EVTTIME_HI,Trace_Out_of_Range_Flag,Trace_Length,Event_Energy


In [112]:
pd.DataFrame(read_events_linear_with_preallocated_numpy())

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12
0,False,5408,0,8,0,0,929243969,0,6912,224,False,11527,0
1,False,4,4,0,2,9,1092930373,2,6367,27,False,0,3831
2,False,4,4,0,2,9,1092947758,0,4970,27,False,0,24377
3,False,4,4,0,2,9,1092964699,1,4535,27,False,0,37858
4,False,4,4,0,2,10,1092964718,7,0,27,False,0,1684
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24593,False,4,4,0,2,10,2093052386,1,2661,27,False,0,16785
24594,False,4,4,0,2,10,2093075227,0,1437,27,False,0,4060
24595,False,4,4,0,2,10,2093081812,7,0,27,False,0,1832
24596,False,4,4,0,2,9,2093109289,2,7719,27,False,0,26102


In [105]:
correct_bit_fields.dtypes

Finish Code                 bool
Event Length               int64
Header Length              int64
CrateID                    int64
SlotID                     int64
Chan#                      int64
EVTTIME_LO                 int64
CFD trigger source bits    int64
CFD Fractional Time        int64
EVTTIME_HI                 int64
Trace Out-of-Range Flag     bool
Trace Length               int64
Event Energy               int64
dtype: object

In [ ]:
np_dtype = 

In [114]:
np.empty(10, formats='b1,i8,i8,i8,i8,i8,i8,i8,i8,i8,b1,i8,i8'.split(','))

TypeError: 'formats' is an invalid keyword argument for empty()

In [106]:
np.empty(10,dtype='b1,i8,i8,i8,i8,i8,i8,i8,i8,i8,b1,i8,i8')


array([(False,                    0,  792633534417207296),
       (False,                    0,                   0),
       (False,                    0,                   0),
       (False,                    0,                   0),
       (False,                    0,                   0),
       (False,                    0,                   0),
       (False,                    0,                   0),
       (False, -9223363266037730281, 5764616293851267076),
       ( True,   247698013766381318,                   0),
       (False,                    0,                   0)],
      dtype=[('f0', '?'), ('f1', '<i8'), ('f2', '<i8')])

In [118]:
np_dtype = np.format_parser(formats='b1,i8,i8,i8,i8,i8,i8,i8,i8,i8,b1,i8,i8'.split(','), names=field_names, titles=[]).dtype

In [120]:
pd.DataFrame(np.empty(10,dtype=np_dtype))

,Finish_Code,Event_Length,Header_Length,CrateID,SlotID,Chan,EVTTIME_LO,CFD_trigger_source_bits,CFD_Fractional_Time,EVTTIME_HI,Trace_Out_of_Range_Flag,Trace_Length,Event_Energy
0,False,0,0,0,0,0,0,0,0,0,False,0,0
1,False,0,0,0,0,0,0,0,0,0,False,0,0
2,False,0,0,0,0,0,0,0,0,0,False,0,0
3,False,0,0,0,0,0,0,0,0,0,False,0,0
4,False,0,0,0,0,0,0,0,0,0,False,0,0
5,False,0,0,0,0,0,0,0,0,0,False,0,0
6,False,0,0,0,0,0,0,0,0,0,False,0,0
7,False,0,0,0,0,0,0,0,0,0,False,0,0
8,False,0,0,0,0,0,0,0,0,0,False,0,0
9,False,0,0,0,0,0,0,0,0,0,False,0,0


In [99]:
empty = pd.DataFrame({
    field_name: pd.Series(index=range(100), dtype=dtype) for field_name, dtype in zip(correct_bit_fields.columns, correct_bit_fields.dtypes)
})

In [100]:
empty.iloc[0] = parse_single_binary_event(bin_data)

In [101]:
empty

,Finish Code,Event Length,Header Length,CrateID,SlotID,Chan#,EVTTIME_LO,CFD trigger source bits,CFD Fractional Time,EVTTIME_HI,Trace Out-of-Range Flag,Trace Length,Event Energy
0,False,4,4,0,2,10,1092838199,7,0,27,False,0,1837
1,False,0,0,0,0,0,0,0,0,0,False,0,0
2,False,0,0,0,0,0,0,0,0,0,False,0,0
3,False,0,0,0,0,0,0,0,0,0,False,0,0
4,False,0,0,0,0,0,0,0,0,0,False,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,False,0,0,0,0,0,0,0,0,0,False,0,0
96,False,0,0,0,0,0,0,0,0,0,False,0,0
97,False,0,0,0,0,0,0,0,0,0,False,0,0
98,False,0,0,0,0,0,0,0,0,0,False,0,0


In [85]:
pd.DataFrame().reindex_like(correct_bit_fields)

,Finish Code,Event Length,Header Length,CrateID,SlotID,Chan#,EVTTIME_LO,CFD trigger source bits,CFD Fractional Time,EVTTIME_HI,Trace Out-of-Range Flag,Trace Length,Event Energy
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24593,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24594,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24595,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24596,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [97]:
correct_bit_fields.dtypes[0].str

'|b1'

In [92]:
correct_bit_fields.dtypes[0].name

'bool'

In [57]:
parse_single_binary_event(bin_data)

(False, 4, 4, 0, 2, 10, 1092838199, 7, 0, 27, False, 0, 1837)

In [67]:
for f in parse_single_binary_event(bin_data):
    print(type(f))

<class 'bool'>
<class 'int'>
<class 'int'>
<class 'int'>
<class 'int'>
<class 'int'>
<class 'int'>
<class 'int'>
<class 'int'>
<class 'int'>
<class 'bool'>
<class 'int'>
<class 'int'>
